# MathFrameworkExperiments
**Purpose:** Strengthening the paper's empirical foundation via:
1. Degeneracy & small-signal rigor (counting fails story)
2. Operator interpretability diagnostics + regime case study
3. Chapman–Kolmogorov consistency as a diagnostic
4. Uncertainty: multi-seed + block bootstrap CIs

**Do NOT modify `notebooks/MasterNotebook.ipynb`.**  
All heavy computation is delegated to `scripts/` modules.

In [1]:
import sys, os
from pathlib import Path

# Allow importing from the repo root scripts/ directory
REPO_ROOT = Path.cwd()
# Handle both "notebooks/" and "current notebooks/" as notebook subdirs
if REPO_ROOT.name in ("notebooks", "current notebooks") or not (REPO_ROOT / "scripts").is_dir():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from scripts.config import make_config, save_config
from scripts.data import load_master_dataset, compute_returns, preprocess_features, build_all_splits, build_all_ck_splits
from scripts.bins import compute_X_t, get_xt_labels_for_ck, build_all_configs, assign_bins, get_edges, compute_sigma
from scripts.models import StateConditionedNet, StateFreeNet
from scripts.train import (
    train_one_run, build_loaders, build_ck_loaders,
    build_A_t_neural, build_A_t_statefree,
    cache_model, load_cached_model, is_cached,
    MasterDataset,
)
from scripts.eval import (
    evaluate_model, evaluate_baselines, mean_log_likelihood,
    compute_degeneracy_stats, compute_transition_sparsity,
    build_ck_composed, compute_ck_errors,
    compute_dobrushin, compute_row_heterogeneity, compute_row_entropy,
    compute_spectral_mixing_proxy,
    compute_pit, compute_ece, compute_brier,
    get_loglik_per_sample_model, get_loglik_per_sample_baseline,
    block_bootstrap_ci,
    _compute_marginal, _build_backoff_matrix, _compute_conditional_additive,
)
from scripts.plotting import (
    save_fig,
    plot_sparsity_vs_N, plot_transition_sparsity_table,
    plot_ck_error_summary, plot_ck_time_series,
    plot_dobrushin_over_time, plot_row_heterogeneity_over_time,
    plot_entropy_over_time, plot_spectral_proxy_over_time,
    plot_At_heatmap_snapshot, plot_regime_panel,
    plot_pit_histogram, plot_reliability_curve,
)

print("Imports OK")

Imports OK


In [2]:
cfg = make_config()
OUT_DIR = Path(cfg.output_dir)
FIG_DIR = OUT_DIR / "figures"
CACHE_DIR = OUT_DIR / "cache"
for d in [OUT_DIR, FIG_DIR, CACHE_DIR, OUT_DIR / "multiasset_edges"]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fixed seeds
SEED = cfg.seed
torch.manual_seed(SEED)
np.random.seed(SEED)

# FAST_MODE: set True for a quick smoke test
FAST_MODE = False
if FAST_MODE:
    cfg.horizons = [1]
    cfg.n_bins_list = [10, 55]
    cfg.ck_horizons = [1]
    cfg.max_epochs = 20
    cfg.seeds = [42]
    print("FAST_MODE active — reduced config")

save_config(cfg, OUT_DIR / "config.yaml")
print(f"Output dir : {OUT_DIR}")
print(f"Device     : {DEVICE}")
print(f"Git hash   : {cfg.git_hash}")
print(f"Date stamp : {cfg.date_stamp}")

Output dir : /Users/JanRovirosaIlla/DeepMarkovResearch/results/paper_upgrade/2026-03-07
Device     : cpu
Git hash   : 2c1cefc
Date stamp : 2026-03-07


In [3]:
prices, F_raw, feature_cols = load_master_dataset("dataset")
n_features = F_raw.shape[1]
print(f"Prices: {len(prices)}, Features: {n_features}")

splits = build_all_splits(prices, cfg.horizons)
# Fit z-score on h=1 train
idx_train_h1 = splits[1]["idx_train"]
F_normed = preprocess_features(F_raw, idx_train_h1)

# Compute X_t (1-day return bins, N_XT=55)
T_1 = splits[1]["T_h"]
train_end_h1 = splits[1]["idx_train"][-1] + 1
X_t_all, N_XT, edges_xt = compute_X_t(prices, cfg.n_xt_target, train_end_h1)
print(f"N_XT={N_XT}, X_t range [{X_t_all.min()}, {X_t_all.max()}]")

# CK-specific splits: T_ck = len(X_t_all) - h  (one fewer than T_h = len(prices) - h)
# because y_ck[h] = X_t_all[h:] has length len(X_t_all) - h
ck_splits = build_all_ck_splits(X_t_all, cfg.ck_horizons)
print("CK splits:", {h: sp["T_ck"] for h, sp in ck_splits.items()})

# CK labels for all horizons (needed in Sections A+B)
ck_labels = get_xt_labels_for_ck(X_t_all, cfg.ck_horizons)

# Build cumulative-return configs (mirrors master notebook)
configs = build_all_configs(
    prices, F_normed, X_t_all,
    cfg.horizons, cfg.n_bins_list, N_XT, edges_xt, splits,
    sigma_anchor=cfg.sigma_anchor,
    results_dir=None,
)

# Load existing master results (no rerun)
master_results = pd.read_csv("results/master_grid_results.csv")
print(f"Loaded master_grid_results.csv: {master_results.shape}")

Prices: 2369, Features: 194
N_XT=55, X_t range [0, 54]
CK splits: {1: 2367, 2: 2366, 5: 2363, 10: 2358}
Loaded master_grid_results.csv: (80, 15)


In [4]:
# Clear stale h=1 cumulative cache files (label definition was corrected)
# CK h=1 weights (ck_*) and A_t arrays are NOT cleared — CK label unchanged
stale_patterns = [
    "calib_state_cond_h1_*",
    "calib_state_free_h1_*",
    "seed_state_cond_h1_*",
    "seed_state_free_h1_*",
    "loglik_state_cond_h1_*",
    "loglik_state_free_h1_*",
    "h1_fresh_*",  # from any prior partial run with corrected label
]

n_cleared = 0
for pat in stale_patterns:
    for f in CACHE_DIR.glob(pat):
        f.unlink()
        print(f"  Cleared: {f.name}")
        n_cleared += 1

if n_cleared == 0:
    print("No stale h=1 cumulative cache files found (already clean).")
else:
    print(f"Cleared {n_cleared} stale h=1 cumulative cache files.")
print("CK h=1 weights preserved.")

No stale h=1 cumulative cache files found (already clean).
CK h=1 weights preserved.


## Section A: Degeneracy & Small-Signal Rigor

**Motivation:** With only ~2,300 training days and 55×55 = 3,025 possible state-to-state 
transitions, count-based methods suffer severe degeneracy. This section quantifies that
degeneracy and motivates the neural regularized approach.

In [5]:
deg_label_rows = []
for h in cfg.horizons:
    sp = splits[h]
    for N in cfg.n_bins_list:
        cfg_key = (h, N)
        if cfg_key not in configs:
            continue
        c = configs[cfg_key]
        y_tr = c["y_all"][sp["idx_train"]]
        N_actual = c["N_actual"]
        stats = compute_degeneracy_stats(y_tr, N_actual, thresholds=tuple(cfg.sparsity_thresh))
        row = {"h": h, "N": N, "N_actual": N_actual,
               "n_train": len(sp["idx_train"]), "effective_bins": stats["effective"]}
        for k, frac in stats["frac_below"].items():
            row[f"frac_below_{k}"] = frac
        deg_label_rows.append(row)

df_deg_label = pd.DataFrame(deg_label_rows)
df_deg_label.to_csv(OUT_DIR / "degeneracy_label_table.csv", index=False)
print("Saved degeneracy_label_table.csv")
print(df_deg_label.to_string(index=False))

Saved degeneracy_label_table.csv
 h  N  N_actual  n_train  effective_bins  frac_below_5  frac_below_10
 1 10        10     1656              10           0.0            0.0
 1 20        20     1656              20           0.0            0.0
 1 35        35     1656              35           0.0            0.0
 1 55        55     1656              55           0.0            0.0
 2 10        10     1656              10           0.0            0.0
 2 20        20     1656              20           0.0            0.0
 2 35        35     1656              35           0.0            0.0
 2 55        55     1656              55           0.0            0.0
 5 10        10     1654              10           0.0            0.0
 5 20        20     1654              20           0.0            0.0
 5 35        35     1654              35           0.0            0.0
 5 55        55     1654              55           0.0            0.0
10 10        10     1650              10           0.0   

In [6]:
sparsity_rows = []
sparsity_data = {}  # for plotting: {(h, N): stats}

CELL_METRICS = [
    "frac_cells_zero",
    "frac_cells_lt5",
    "frac_cells_lt10",
    "frac_rows_lt5",
    "frac_rows_lt10",
    "median_nonzero_per_row",
    "p90_nonzero_per_row",
    "median_row_entropy_empirical",
    "median_row_maxprob_empirical",
]

# Part A: cumulative configs
for h in cfg.horizons:
    sp = splits[h]
    T_h = sp["T_h"]
    idx_tr = sp["idx_train"]
    for N in cfg.n_bins_list:
        cfg_key = (h, N)
        if cfg_key not in configs:
            continue
        c = configs[cfg_key]
        N_actual = c["N_actual"]
        X_aligned = X_t_all[:T_h][idx_tr]
        Y_aligned = c["y_all"][:T_h][idx_tr]
        # Leakage check: h=1 label must NOT be identical to state after label fix
        if h == 1:
            frac_eq = float(np.mean(X_aligned == Y_aligned))
            assert frac_eq < 0.99, (
                f"LEAKAGE at h=1, N={N}: fraction(X_t==Y_t) = {frac_eq:.4f}. "
                "Expected < 0.99 with corrected label (Y_t = next-day return after X_t)."
            )
            if N == cfg.n_bins_list[-1]:
                print(f"  Leakage check h=1 N={N}: fraction(X_t==Y_t) = {frac_eq:.4f} (OK)")
        stats = compute_transition_sparsity(X_aligned, Y_aligned, N_XT, N_actual)
        sparsity_data[(h, N)] = stats
        row = {"config_type": "cumulative", "h": h, "N": N, "N_actual": N_actual}
        for m in CELL_METRICS:
            row[m] = stats[m]
        sparsity_rows.append(row)

# Part B: CK configs (N_output = N_XT = 55, label = X_{t+h})
# Use CK-specific splits so indices are capped to T_ck = len(X_t_all) - h
for h in cfg.ck_horizons:
    sp_ck = ck_splits[h]
    T_ck = sp_ck["T_ck"]
    idx_tr = sp_ck["idx_train"]
    y_ck = ck_labels[h]
    X_aligned = X_t_all[:T_ck][idx_tr]
    Y_aligned  = y_ck[idx_tr]
    stats = compute_transition_sparsity(X_aligned, Y_aligned, N_XT, N_XT)
    sparsity_data[(h, "ck")] = stats
    row = {"config_type": "ck", "h": h, "N": N_XT, "N_actual": N_XT}
    for m in CELL_METRICS:
        row[m] = stats[m]
    sparsity_rows.append(row)

df_sparsity = pd.DataFrame(sparsity_rows)
df_sparsity.to_csv(OUT_DIR / "degeneracy_transition_table.csv", index=False)
print("Saved degeneracy_transition_table.csv")
print(df_sparsity.to_string(index=False))

  Leakage check h=1 N=55: fraction(X_t==Y_t) = 0.0229 (OK)
Saved degeneracy_transition_table.csv
config_type  h  N  N_actual  frac_cells_zero  frac_cells_lt5  frac_cells_lt10  frac_rows_lt5  frac_rows_lt10  median_nonzero_per_row  p90_nonzero_per_row  median_row_entropy_empirical  median_row_maxprob_empirical
 cumulative  1 10        10         0.056364        0.827273         0.994545            0.0             0.0                    10.0                 10.0                      2.130573                      0.200000
 cumulative  1 20        20         0.224545        0.980000         0.999091            0.0             0.0                    16.0                 17.0                      2.626957                      0.133333
 cumulative  1 35        35         0.436364        0.995844         1.000000            0.0             0.0                    20.0                 22.0                      2.880478                      0.100000
 cumulative  1 55        55         0.579504   

In [7]:
thresh = min(cfg.sparsity_thresh)
fig_a = plot_sparsity_vs_N(sparsity_data, cfg.horizons, cfg.n_bins_list, thresh, FIG_DIR)
plt.close(fig_a)
fig_b = plot_transition_sparsity_table(df_sparsity, thresh, FIG_DIR)
plt.close(fig_b)
print("Section A figures saved.")

Section A figures saved.


## Section B: Chapman–Kolmogorov Consistency (Diagnostic)

**Label definition:** y_t^(h) := X_{t+h} (the 1-day return bin h steps ahead).  
This is NOT the cumulative h-step return. It places all A_t^(h) in the same 55×55 state space,
making matrix multiplication valid.

**CK test (time-inhomogeneous):**  
A_composed_t^(h) = A_t^(1) × A_{t+1}^(1) × ... × A_{t+h-1}^(1)  
Compare against directly predicted A_t^(h). Metrics: mean KL, mean TV, Frobenius.

**Note:** StateFree A_t has degenerate dynamics (all rows identical at each t).  
This is expected and means StateFree CK error reflects purely time-driven dynamics.

**Why CK may fail here:** 2,369 days → 3,025 possible state-to-state transitions (severe degeneracy);
weak state signal (low MI); time-inhomogeneity from regime changes; discretization artifacts.

In [8]:
ck_models = {}   # {(model_type, h): model}
A_t_ck = {}     # {(model_type, h): np.ndarray (T_ck, 55, 55)}

for h in cfg.ck_horizons:
    # Use CK-specific splits: T_ck = len(X_t_all) - h
    sp_ck = ck_splits[h]
    T_ck  = sp_ck["T_ck"]
    idx_tr = sp_ck["idx_train"]
    idx_va = sp_ck["idx_val"]
    idx_te = sp_ck["idx_test"]
    y_ck = ck_labels[h]   # length T_ck

    # Build data loaders for CK task
    train_loader, val_loader, test_loader = build_ck_loaders(
        F_normed, X_t_all, y_ck, idx_tr, idx_va, idx_te,
        batch_train=cfg.batch_train, batch_eval=cfg.batch_eval,
    )

    # Sigma for CK task (output = 55 bins, same space as input)
    R_h = compute_returns(prices, h)
    R_tr = R_h[idx_tr]
    _, edges_h = get_edges(R_tr, N_XT)
    sigma_ck = compute_sigma(edges_h, edges_xt, cfg.sigma_anchor)

    for model_type in ["state_cond", "state_free"]:
        weight_path = CACHE_DIR / f"ck_{model_type}_h{h}_seed{SEED}.pt"
        A_path = CACHE_DIR / f"A_t_ck_{model_type}_h{h}.npy"

        if model_type == "state_cond":
            model = StateConditionedNet(n_features, N_XT, N_XT,
                                        hidden_dims=cfg.hidden_dims, dropout=cfg.dropout)
        else:
            model = StateFreeNet(n_features, N_XT,
                                  hidden_dims=cfg.hidden_dims, dropout=cfg.dropout)

        torch.manual_seed(SEED)
        if is_cached(weight_path):
            load_cached_model(model, weight_path)
            print(f"Loaded cached CK model: {model_type} h={h}")
        else:
            print(f"Training CK model: {model_type} h={h} (T_ck={T_ck})...")
            best_state, _ = train_one_run(
                model, train_loader, val_loader, N_XT, sigma_ck, DEVICE,
                lr=cfg.lr, weight_decay=cfg.weight_decay,
                max_epochs=cfg.max_epochs, patience=cfg.patience,
                grad_clip=cfg.grad_clip, verbose=True,
            )
            cache_model(best_state, weight_path)

        model.eval()
        ck_models[(model_type, h)] = model

        if is_cached(A_path):
            A_t = np.load(A_path)
            print(f"Loaded cached A_t: {model_type} h={h}, shape={A_t.shape}")
        else:
            print(f"Building A_t matrices: {model_type} h={h} ...")
            # Build A_t for the CK-valid time range [0, T_ck)
            full_indices = np.arange(T_ck)
            if model_type == "state_cond":
                A_t = build_A_t_neural(model, F_normed, full_indices, N_XT, N_XT, DEVICE)
            else:
                A_t = build_A_t_statefree(model, F_normed, full_indices, N_XT, N_XT, DEVICE)
            np.save(A_path, A_t)
            print(f"  Saved A_t shape={A_t.shape}")

        A_t_ck[(model_type, h)] = A_t

print("Section B: all CK models and A_t matrices ready.")

Training CK model: state_cond h=1 (T_ck=2367)...


  epoch   1: train=2.6154 val=2.6070 t_acc=0.021 v_acc=0.006


  epoch  20: train=2.4913 val=2.5774 t_acc=0.045 v_acc=0.025
  Early stop at epoch 21
Building A_t matrices: state_cond h=1 ...


  Saved A_t shape=(2367, 55, 55)
Training CK model: state_free h=1 (T_ck=2367)...
  epoch   1: train=2.6171 val=2.6085 t_acc=0.018 v_acc=0.008


  Early stop at epoch 18
Building A_t matrices: state_free h=1 ...
  Saved A_t shape=(2367, 55, 55)
Training CK model: state_cond h=2 (T_ck=2366)...


...

  epoch  20: train=3.4281 val=3.5311 t_acc=0.042 v_acc=0.023
  Early stop at epoch 21
Building A_t matrices: state_free h=5 ...
  Saved A_t shape=(2363, 55, 55)
Training CK model: state_cond h=10 (T_ck=2358)...


  epoch   1: train=3.9401 val=3.9367 t_acc=0.018 v_acc=0.028


  Early stop at epoch 11
Building A_t matrices: state_cond h=10 ...


  Saved A_t shape=(2358, 55, 55)
Training CK model: state_free h=10 (T_ck=2358)...
  epoch   1: train=3.9393 val=3.9408 t_acc=0.019 v_acc=0.017


  Early stop at epoch 12
Building A_t matrices: state_free h=10 ...
  Saved A_t shape=(2358, 55, 55)
Section B: all CK models and A_t matrices ready.


In [9]:
ck_table_rows = []
ck_time_dict = {}
for h in cfg.ck_horizons:
    # Use CK-specific splits
    sp_ck = ck_splits[h]
    T_ck  = sp_ck["T_ck"]
    idx_tr = sp_ck["idx_train"]
    idx_te = sp_ck["idx_test"]
    y_ck = ck_labels[h]   # length T_ck

    # CK backoff baseline on X_t -> X_{t+h} (55x55)
    X_tr = X_t_all[:T_ck][idx_tr]
    Y_tr = y_ck[idx_tr]
    marginal_ck = _compute_marginal(Y_tr, N_XT)
    best_bk_ll, best_alpha_bk, best_tau_bk = -np.inf, None, None
    X_va = X_t_all[:T_ck][sp_ck["idx_val"]]
    Y_va = y_ck[sp_ck["idx_val"]]
    for alpha in cfg.alpha_grid:
        for tau in cfg.tau_grid:
            A_bk, _, _ = _build_backoff_matrix(X_tr, Y_tr, N_XT, N_XT, alpha, tau, marginal_ck)
            ll = mean_log_likelihood(A_bk[X_va], Y_va)
            if ll > best_bk_ll:
                best_bk_ll, best_alpha_bk, best_tau_bk = ll, alpha, tau
    A_bk_ck, _, _ = _build_backoff_matrix(X_tr, Y_tr, N_XT, N_XT,
                                            best_alpha_bk, best_tau_bk, marginal_ck)

    # State weights (train visitation)
    state_counts = np.bincount(X_tr, minlength=N_XT).astype(np.float64)

    # Build A_t arrays on test window (slice from full A_t arrays)
    test_start = idx_te[0]
    test_end   = idx_te[-1] + 1
    # For CK composition, need h steps starting at each test point
    max_test_for_ck = T_ck - h + 1  # last valid start for h-step composition

    models_to_test = {
        "state_cond": A_t_ck[("state_cond", h)],
        "state_free":  A_t_ck[("state_free", h)],
    }

    for model_name, A_full in models_to_test.items():
        # A_full: (T_ck, 55, 55)
        A1_full = A_t_ck[(model_name, 1)] if 1 in cfg.ck_horizons else None

        if h == 1:
            # CK trivially satisfied for h=1 (compare A^(1) with itself)
            n_te = len(idx_te)
            ck_time_dict[(model_name, h)] = np.zeros(n_te)
            ck_table_rows.append({
                "model": model_name, "h": h,
                "mean_kl": 0.0, "mean_tv": 0.0, "frobenius": 0.0,
                "note": "h=1: trivial (identity composition)",
            })
            continue

        if A1_full is None:
            print(f"  Skipping CK for h={h} (h=1 model not available)")
            continue

        # Align test window: cap at max_test_for_ck
        valid_te_end = min(test_end, max_test_for_ck)
        if valid_te_end <= test_start:
            print(f"  Skipping h={h} model={model_name}: test window too small for CK")
            continue

        A_h_te  = A_full[test_start:valid_te_end]          # direct h-step
        # For composition we need A^(1) at t, t+1, ..., t+h-1
        # A1_full has length T_ck for h=1, but T_ck(h=1) >= T_ck(h) + h - 1
        n_needed = valid_te_end + h - 1
        A1_window = A1_full[test_start:min(n_needed, len(A1_full))]
        A_comp = build_ck_composed(A1_window, h)    # (len(A1_window) - h + 1, 55, 55)

        n_common = min(len(A_h_te), len(A_comp))
        if n_common == 0:
            print(f"  Skipping h={h} model={model_name}: no common time steps")
            continue
        errors = compute_ck_errors(A_h_te[:n_common], A_comp[:n_common], state_counts)
        ck_time_dict[(model_name, h)] = errors["per_time_kl"]
        ck_table_rows.append({
            "model": model_name, "h": h,
            "mean_kl": errors["mean_kl"],
            "mean_tv": errors["mean_tv"],
            "frobenius": errors["frobenius"],
        })
        print(f"  CK h={h} {model_name}: KL={errors['mean_kl']:.4f} TV={errors['mean_tv']:.4f}")

    # Backoff: build its A_t by expanding matrix (same matrix for all t)
    A_bk_expanded = np.tile(A_bk_ck[None], (T_ck, 1, 1))
    if h > 1:
        valid_te_end_bk = min(test_end, max_test_for_ck)
        if valid_te_end_bk > test_start:
            A_comp_bk = build_ck_composed(
                A_bk_expanded[test_start:min(valid_te_end_bk + h - 1, T_ck)], h
            )
            A_h_bk = A_bk_expanded[test_start:valid_te_end_bk]
            n_common = min(len(A_h_bk), len(A_comp_bk))
            if n_common > 0:
                errors_bk = compute_ck_errors(A_h_bk[:n_common], A_comp_bk[:n_common], state_counts)
                ck_time_dict[("backoff_ck", h)] = errors_bk["per_time_kl"]
                ck_table_rows.append({
                    "model": "backoff_ck", "h": h,
                    "mean_kl": errors_bk["mean_kl"],
                    "mean_tv": errors_bk["mean_tv"],
                    "frobenius": errors_bk["frobenius"],
                })

    # --- backoff_ck h=1 trivial row ---
    if h == 1:
        ck_table_rows.append({
            "model": "backoff_ck", "h": h,
            "mean_kl": 0.0, "mean_tv": 0.0, "frobenius": 0.0,
            "note": "h=1: trivial (identity composition)",
        })
    # (h>1 backoff_ck rows are appended by the 'Backoff: build its A_t' block above)


df_ck = pd.DataFrame(ck_table_rows)
df_ck.to_csv(OUT_DIR / "ck_table.csv", index=False)
print("Saved ck_table.csv")
print(df_ck.to_string(index=False))

  CK h=2 state_cond: KL=0.1518 TV=0.1884
  CK h=2 state_free: KL=0.0299 TV=0.0868
  CK h=5 state_cond: KL=0.1403 TV=0.2060
  CK h=5 state_free: KL=0.0225 TV=0.0854


  CK h=10 state_cond: KL=0.1517 TV=0.1930
  CK h=10 state_free: KL=0.0295 TV=0.0847
Saved ck_table.csv
     model  h  mean_kl  mean_tv  frobenius                                note
state_cond  1 0.000000 0.000000   0.000000 h=1: trivial (identity composition)
state_free  1 0.000000 0.000000   0.000000 h=1: trivial (identity composition)
backoff_ck  1 0.000000 0.000000   0.000000 h=1: trivial (identity composition)
state_cond  2 0.151826 0.188423   0.439991                                 NaN
state_free  2 0.029924 0.086831   0.225266                                 NaN
backoff_ck  2 0.010630 0.063744   0.150172                                 NaN
backoff_ck  2 0.010699 0.063866   0.150704                                    
state_cond  5 0.140333 0.205968   0.488040                                 NaN
state_free  5 0.022479 0.085442   0.211945                                 NaN
backoff_ck  5 0.002853 0.032793   0.076698                                 NaN
backoff_ck  5 0.002909 0.032

In [10]:
print("Stationarity probe (NOT a CK test): (A_avg^(1))^h vs direct A^(h)")
stationarity_rows = []
for h in cfg.ck_horizons:
    if h == 1 or ("state_cond", 1) not in A_t_ck:
        continue
    sp_ck_h = ck_splits[h]
    sp_ck_1 = ck_splits[1]
    A1_full = A_t_ck[("state_cond", 1)]
    # Average A^(1) over h=1 train window
    A1_train = A1_full[sp_ck_1["idx_train"]]
    A_avg = A1_train.mean(axis=0)  # (55, 55)

    import numpy.linalg as nla
    A_avg_h = nla.matrix_power(A_avg, h)

    # Compare with direct h-step prediction on h-step test window
    idx_te = sp_ck_h["idx_test"]
    A_h_direct = A_t_ck[("state_cond", h)][idx_te]
    A_avg_h_tiled = np.tile(A_avg_h[None], (len(A_h_direct), 1, 1))
    errors_stat = compute_ck_errors(A_h_direct, A_avg_h_tiled)
    stationarity_rows.append({
        "h": h, "mean_kl": errors_stat["mean_kl"],
        "note": "Stationarity probe (NOT CK) — avg A^(1) raised to power h"
    })
    print(f"  h={h}: stationarity probe KL={errors_stat['mean_kl']:.4f}")

df_stat = pd.DataFrame(stationarity_rows)
if len(df_stat):
    df_stat.to_csv(OUT_DIR / "stationarity_probe.csv", index=False)

Stationarity probe (NOT a CK test): (A_avg^(1))^h vs direct A^(h)
  h=2: stationarity probe KL=0.0030
  h=5: stationarity probe KL=0.0488
  h=10: stationarity probe KL=0.0050


In [11]:
if len(df_ck) > 0:
    fig_ck = plot_ck_error_summary(df_ck, FIG_DIR)
    plt.close(fig_ck)
if ck_time_dict:
    fig_ck_t = plot_ck_time_series(ck_time_dict, FIG_DIR)
    plt.close(fig_ck_t)
print("Section B figures saved.")

Section B figures saved.


## Section C: Operator Interpretability Diagnostics

Using the CK h=1 models, we compute A_t^(1) over the full time series and extract
measurable diagnostics:

- **Dobrushin coefficient** δ(A_t): contraction measure
- **Row heterogeneity** ρ(A_t): average pairwise TV between rows — state-dependence strength
- **Row entropy** H(A_t): diffuseness of transitions
- **Spectral mixing proxy**: σ_max of lazy deflated operator (NaN-safe)

We then identify 2–3 regime windows and show A_t snapshots.

In [12]:
diag_series = {}  # {model_type: {"dobrushin": array, ...}}

for model_type in ["state_cond", "state_free"]:
    if ("state_cond", 1) not in A_t_ck and model_type == "state_cond":
        continue
    if ("state_free", 1) not in A_t_ck and model_type == "state_free":
        continue

    A_full = A_t_ck[(model_type, 1)]  # (T_h, 55, 55)
    print(f"Computing diagnostics for {model_type} ({len(A_full)} time steps)...")

    dob  = compute_dobrushin(A_full)
    rhet = compute_row_heterogeneity(A_full)
    rent = compute_row_entropy(A_full)
    spec = compute_spectral_mixing_proxy(A_full)

    diag_series[model_type] = {
        "dobrushin": dob,
        "row_heterogeneity": rhet,
        "row_entropy": rent,
        "spectral_proxy": spec,
    }
    pct_finite = np.isfinite(spec).mean() * 100
    print(f"  Dobrushin:  mean={dob.mean():.4f}  max={dob.max():.4f}")
    print(f"  RowHet:     mean={rhet.mean():.4f}")
    print(f"  RowEntropy: mean={rent.mean():.4f}")
    print(f"  SpectralProxy: {pct_finite:.1f}% finite, mean={np.nanmean(spec):.4f}")

Computing diagnostics for state_cond (2367 time steps)...


  Dobrushin:  mean=0.0294  max=0.0567
  RowHet:     mean=0.0073
  RowEntropy: mean=3.9260
  SpectralProxy: 100.0% finite, mean=0.0138
Computing diagnostics for state_free (2367 time steps)...


  Dobrushin:  mean=0.0000  max=0.0000
  RowHet:     mean=0.0000
  RowEntropy: mean=3.9520
  SpectralProxy: 100.0% finite, mean=0.0000


In [13]:
# Regime detection: top-3 peaks in state_cond dobrushin
if "state_cond" in diag_series:
    dob_series = diag_series["state_cond"]["dobrushin"]
    T_full = len(dob_series)

    # Simple peak detection: top-3 local maxima
    from scipy.signal import find_peaks
    peaks, _ = find_peaks(dob_series, distance=30)
    top_peaks = peaks[np.argsort(dob_series[peaks])[::-1][:3]] if len(peaks) >= 3 else peaks
    top_peaks = sorted(top_peaks)

    # Define regime windows: ±15 days around each peak
    regime_windows = []
    for pk in top_peaks:
        start = max(0, pk - 15)
        end   = min(T_full - 1, pk + 15)
        regime_windows.append((start, end, f"t={pk}"))

    # Snapshot heatmaps
    for model_type in ["state_cond", "state_free"]:
        if (model_type, 1) not in A_t_ck:
            continue
        A_full = A_t_ck[(model_type, 1)]
        for pk in top_peaks:
            if pk < len(A_full):
                plot_At_heatmap_snapshot(A_full[pk], f"t{pk}", FIG_DIR, model_name=model_type)
                plt.close("all")

    print(f"Regime windows: {regime_windows}")
else:
    regime_windows = []
    print("No state_cond diagnostics available — skipping regime analysis.")

Regime windows: [(np.int64(193), np.int64(223), 't=208'), (np.int64(238), np.int64(268), 't=253'), (np.int64(282), np.int64(312), 't=297')]


In [14]:
dob_dict  = {k: v["dobrushin"]        for k, v in diag_series.items()}
rhet_dict = {k: v["row_heterogeneity"] for k, v in diag_series.items()}
rent_dict = {k: v["row_entropy"]       for k, v in diag_series.items()}
spec_dict = {k: v["spectral_proxy"]    for k, v in diag_series.items()}

for plot_fn, data, name in [
    (plot_dobrushin_over_time, dob_dict, "dobrushin"),
    (plot_row_heterogeneity_over_time, rhet_dict, "row_het"),
    (plot_entropy_over_time, rent_dict, "entropy"),
]:
    fig = plot_fn(data, FIG_DIR, regime_windows=regime_windows)
    plt.close(fig)

fig_spec = plot_spectral_proxy_over_time(spec_dict, FIG_DIR, regime_windows=regime_windows)
if fig_spec:
    plt.close(fig_spec)

# Regime panel
if diag_series:
    # Build combined diagnostics dataframe
    rows = []
    for model_type, d in diag_series.items():
        T_m = len(d["dobrushin"])
        for t in range(T_m):
            rows.append({
                "time_idx": t, "model": model_type,
                "dobrushin": d["dobrushin"][t],
                "row_heterogeneity": d["row_heterogeneity"][t],
                "row_entropy": d["row_entropy"][t],
            })
    diag_df = pd.DataFrame(rows)
    fig_rp = plot_regime_panel(diag_df, regime_windows, list(diag_series.keys()), FIG_DIR)
    plt.close(fig_rp)

print("Section C figures saved.")

Section C figures saved.


## Section D: Calibration

We recompute predicted probabilities on the test set for configs (h=1, N=55) and (h=10, N=55).

**master_grid_results.csv contains only scalar NLL — not predicted distributions —
so calibration CANNOT be computed from it; fresh forward passes are required.**

Cached model weights are loaded; if not present, those two configs are retrained.

In [15]:
calib_configs = [(1, 55), (10, 55)]
calib_rows = []

for (h, N) in calib_configs:
    if (h, N) not in configs:
        print(f"Config (h={h}, N={N}) not in configs, skipping calibration.")
        continue
    c = configs[(h, N)]
    N_actual = c["N_actual"]
    sp = splits[h]

    train_loader, val_loader, test_loader = build_loaders(
        c, F_normed, X_t_all,
        batch_train=cfg.batch_train, batch_eval=cfg.batch_eval,
    )

    y_te = c["y_all"][sp["idx_test"]]
    X_te = X_t_all[sp["idx_test"]]

    for model_type in ["state_cond", "state_free"]:
        weight_path = CACHE_DIR / f"calib_{model_type}_h{h}_N{N}_seed{SEED}.pt"

        if model_type == "state_cond":
            model = StateConditionedNet(n_features, N_XT, N_actual,
                                        hidden_dims=cfg.hidden_dims, dropout=cfg.dropout)
        else:
            model = StateFreeNet(n_features, N_actual,
                                  hidden_dims=cfg.hidden_dims, dropout=cfg.dropout)

        torch.manual_seed(SEED)
        if is_cached(weight_path):
            load_cached_model(model, weight_path)
        else:
            print(f"  Retraining calibration model: {model_type} h={h} N={N} ...")
            best_state, _ = train_one_run(
                model, train_loader, val_loader, N_actual, c["sigma"], DEVICE,
                lr=cfg.lr, weight_decay=cfg.weight_decay,
                max_epochs=cfg.max_epochs, patience=cfg.patience,
                grad_clip=cfg.grad_clip,
            )
            cache_model(best_state, weight_path)

        model.eval()
        # Collect full test probs
        all_probs = []
        with torch.no_grad():
            for F_b, xt_b, y_b in test_loader:
                logits = model(F_b.to(DEVICE), xt_b.to(DEVICE))
                probs = torch.softmax(logits, dim=1).cpu().numpy()
                all_probs.append(probs)
        all_probs = np.vstack(all_probs)

        # Event: negative return = bin < N_actual // 2
        event_fn = lambda y, N=N_actual: y < N // 2

        pit_vals = compute_pit(all_probs, y_te)
        ece, conf_b, acc_b = compute_ece(all_probs, y_te, event_fn)
        brier = compute_brier(all_probs, y_te, event_fn)

        calib_rows.append({
            "h": h, "N": N, "model": model_type,
            "ece_neg_return": ece, "brier_neg_return": brier,
        })

        # Save PIT histogram
        fig_pit = plot_pit_histogram(pit_vals, f"{model_type}_h{h}_N{N}", FIG_DIR)
        plt.close(fig_pit)
        # Save reliability curve
        fig_rel = plot_reliability_curve(conf_b, acc_b, ece,
                                         f"negative_return_h{h}_N{N}_{model_type}", FIG_DIR)
        plt.close(fig_rel)

    # Backoff baseline calibration
    y_tr = c["y_all"][sp["idx_train"]]
    X_tr = X_t_all[sp["idx_train"]]
    marginal = _compute_marginal(y_tr, N_actual)
    A_bk, _, _ = _build_backoff_matrix(X_tr, y_tr, N_XT, N_actual,
                                        1e-4, 100, marginal)
    probs_bk = A_bk[X_te]
    event_fn_bk = lambda y, N=N_actual: y < N // 2
    ece_bk, _, _ = compute_ece(probs_bk, y_te, event_fn_bk)
    brier_bk = compute_brier(probs_bk, y_te, event_fn_bk)
    calib_rows.append({
        "h": h, "N": N, "model": "backoff",
        "ece_neg_return": ece_bk, "brier_neg_return": brier_bk,
    })

df_calib = pd.DataFrame(calib_rows)
df_calib.to_csv(OUT_DIR / "calibration_table.csv", index=False)
print("Saved calibration_table.csv")
print(df_calib.to_string(index=False))

  Retraining calibration model: state_cond h=1 N=55 ...


  Retraining calibration model: state_free h=1 N=55 ...


  Retraining calibration model: state_cond h=10 N=55 ...


  Retraining calibration model: state_free h=10 N=55 ...


Saved calibration_table.csv
 h  N      model  ece_neg_return  brier_neg_return
 1 55 state_cond        0.079298          0.246790
 1 55 state_free        0.094828          0.250977
 1 55    backoff        0.080338          0.248459
10 55 state_cond        0.111360          0.248173
10 55 state_free        0.110171          0.248427
10 55    backoff        0.111183          0.248061


## Section E: Robustness & Uncertainty

- 3-seed sweep for key configs: (h=1, N=55) and (h=10, N=55)
- Block bootstrap CIs on per-sample test log-likelihood (key configs only)
- main_results_table.csv: NLL + ΔNLL + CI for all configs (CI=NaN for non-key configs)

In [16]:
seed_results = []  # {h, N, model_type, seed, test_ll, delta_ll}

key_configs = cfg.bootstrap_key_configs  # e.g., [(1, 55), (10, 55)]

for (h, N) in key_configs:
    if (h, N) not in configs:
        continue
    c = configs[(h, N)]
    N_actual = c["N_actual"]
    sp = splits[h]

    train_loader, val_loader, test_loader = build_loaders(
        c, F_normed, X_t_all,
        batch_train=cfg.batch_train, batch_eval=cfg.batch_eval,
    )

    for seed in cfg.seeds:
        for model_type in ["state_cond", "state_free"]:
            weight_path = CACHE_DIR / f"seed_{model_type}_h{h}_N{N}_seed{seed}.pt"
            loglik_path = CACHE_DIR / f"loglik_{model_type}_h{h}_N{N}_seed{seed}.npy"

            if model_type == "state_cond":
                model = StateConditionedNet(n_features, N_XT, N_actual,
                                            hidden_dims=cfg.hidden_dims, dropout=cfg.dropout)
            else:
                model = StateFreeNet(n_features, N_actual,
                                      hidden_dims=cfg.hidden_dims, dropout=cfg.dropout)

            torch.manual_seed(seed)
            np.random.seed(seed)

            if is_cached(weight_path):
                load_cached_model(model, weight_path)
            else:
                print(f"  Training: {model_type} h={h} N={N} seed={seed} ...")
                best_state, _ = train_one_run(
                    model, train_loader, val_loader, N_actual, c["sigma"], DEVICE,
                    lr=cfg.lr, weight_decay=cfg.weight_decay,
                    max_epochs=cfg.max_epochs, patience=cfg.patience,
                    grad_clip=cfg.grad_clip,
                )
                cache_model(best_state, weight_path)

            if is_cached(loglik_path):
                lp = np.load(loglik_path)
            else:
                lp = get_loglik_per_sample_model(model, test_loader, N_actual, DEVICE)
                np.save(loglik_path, lp)

            seed_results.append({
                "h": h, "N": N, "model": model_type, "seed": seed,
                "test_ll": float(lp.mean()),
                "loglik_per_sample": lp,
            })

df_seeds = pd.DataFrame([{k: v for k, v in r.items() if k != "loglik_per_sample"}
                          for r in seed_results])
print("Multi-seed results:")
print(df_seeds.groupby(["h", "N", "model"])[["test_ll"]].agg(["mean", "std"]).to_string())

  Training: state_cond h=1 N=55 seed=42 ...


  Training: state_free h=1 N=55 seed=42 ...


  Training: state_cond h=1 N=55 seed=7 ...


  Training: state_free h=1 N=55 seed=7 ...


  Training: state_cond h=1 N=55 seed=123 ...


...

  Training: state_cond h=10 N=55 seed=7 ...


  Training: state_free h=10 N=55 seed=7 ...


  Training: state_cond h=10 N=55 seed=123 ...


  Training: state_free h=10 N=55 seed=123 ...


Multi-seed results:
                   test_ll          
                      mean       std
h  N  model                         
1  55 state_cond -4.040218  0.028358
      state_free -4.048051  0.043824
10 55 state_cond -4.001862  0.012398
      state_free -3.990675  0.005392


In [17]:
# Load master results for h>=2 only (h=1 rows are stale: label definition corrected)
mr = master_results[master_results["h"] >= 2].copy()

# Marginal LL per (h, N) for h>=2 (from master grid)
marginal_ll = mr[mr["model"] == "marginal"].set_index(["h", "N"])["test_ll"].to_dict()

# --- Fresh h=1 results with corrected label ---
print("Computing fresh h=1 baseline results (corrected label)...")
h1_baseline_res = {}
for N in cfg.n_bins_list:
    if (1, N) not in configs:
        continue
    c = configs[(1, N)]
    bres = evaluate_baselines(c, X_t_all, N_XT, cfg.alpha_grid, cfg.tau_grid)
    h1_baseline_res[(1, N)] = bres
    marginal_ll[(1, N)] = bres["marginal"]["test_ll"]
    print(f"  h=1 N={N}: marginal={bres['marginal']['test_ll']:.4f}, "
          f"additive={bres['additive']['test_ll']:.4f}, "
          f"backoff={bres['backoff']['test_ll']:.4f}")

# --- Build main results table ---
main_rows = []

# Part A: h>=2 rows from master grid (label definition unchanged for h>=2)
for _, row in mr.iterrows():
    h, N, model = int(row["h"]), int(row["N"]), row["model"]
    ci_lower, ci_upper = np.nan, np.nan
    if (h, N) in key_configs and model in ["state_cond_nn", "state_free_nn"]:
        m_type = "state_cond" if model == "state_cond_nn" else "state_free"
        lp_list = [r["loglik_per_sample"] for r in seed_results
                   if r["h"] == h and r["N"] == N and r["model"] == m_type]
        if lp_list:
            lp_avg = np.stack(lp_list).mean(axis=0)
            _, ci_lower, ci_upper = block_bootstrap_ci(
                lp_avg, cfg.boot_block_size, cfg.n_boot, seed=SEED
            )
    delta = row["test_ll"] - marginal_ll.get((h, N), np.nan)
    main_rows.append({
        "h": h, "N": N, "model": model,
        "test_ll": row["test_ll"],
        "delta_ll": delta,
        "val_ll": row.get("val_ll", np.nan),
        "accuracy": row.get("accuracy", np.nan),
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
    })

# Part B: h=1 rows (freshly computed with corrected label)
for N in cfg.n_bins_list:
    if (1, N) not in configs:
        continue
    c = configs[(1, N)]
    N_actual = c["N_actual"]
    bres = h1_baseline_res[(1, N)]
    marg_ll_h1 = marginal_ll[(1, N)]

    # Baselines
    for m_name, m_res in bres.items():
        delta = m_res["test_ll"] - marg_ll_h1
        main_rows.append({
            "h": 1, "N": N, "model": m_name,
            "test_ll": m_res["test_ll"],
            "delta_ll": delta,
            "val_ll": m_res.get("val_ll", np.nan),
            "accuracy": m_res.get("accuracy", np.nan),
            "ci_lower": np.nan,
            "ci_upper": np.nan,
        })

    # Neural models
    for model_type in ["state_cond", "state_free"]:
        m_name = f"{model_type}_nn"
        ci_lower, ci_upper = np.nan, np.nan

        # Use seed_results for key configs (h=1, N=55 trained in Section E)
        sr = [r for r in seed_results
              if r["h"] == 1 and r["N"] == N and r["model"] == model_type]
        if sr:
            mean_ll = float(np.mean([r["test_ll"] for r in sr]))
            if (1, N) in key_configs:
                lp_list = [r["loglik_per_sample"] for r in sr]
                lp_avg = np.stack(lp_list).mean(axis=0)
                _, ci_lower, ci_upper = block_bootstrap_ci(
                    lp_avg, cfg.boot_block_size, cfg.n_boot, seed=SEED
                )
        else:
            # Train single-seed fresh model for non-key (h=1, N<55) configs
            train_loader, val_loader, test_loader = build_loaders(
                c, F_normed, X_t_all,
                batch_train=cfg.batch_train, batch_eval=cfg.batch_eval,
            )
            weight_path = CACHE_DIR / f"h1_fresh_{model_type}_N{N}_seed{SEED}.pt"
            if model_type == "state_cond":
                m = StateConditionedNet(n_features, N_XT, N_actual,
                                        hidden_dims=cfg.hidden_dims, dropout=cfg.dropout)
            else:
                m = StateFreeNet(n_features, N_actual,
                                  hidden_dims=cfg.hidden_dims, dropout=cfg.dropout)
            torch.manual_seed(SEED)
            if is_cached(weight_path):
                load_cached_model(m, weight_path)
                test_loader_fresh = build_loaders(
                    c, F_normed, X_t_all,
                    batch_train=cfg.batch_train, batch_eval=cfg.batch_eval,
                )[2]
            else:
                print(f"  Training h=1 N={N} {model_type} (corrected label)...")
                best_state, _ = train_one_run(
                    m, train_loader, val_loader, N_actual, c["sigma"], DEVICE,
                    lr=cfg.lr, weight_decay=cfg.weight_decay,
                    max_epochs=cfg.max_epochs, patience=cfg.patience,
                    grad_clip=cfg.grad_clip,
                )
                cache_model(best_state, weight_path)
                test_loader_fresh = test_loader
            res = evaluate_model(m, test_loader_fresh, N_actual, DEVICE)
            mean_ll = res["mean_ll"]

        delta = mean_ll - marg_ll_h1
        main_rows.append({
            "h": 1, "N": N, "model": m_name,
            "test_ll": mean_ll,
            "delta_ll": delta,
            "val_ll": np.nan,
            "accuracy": np.nan,
            "ci_lower": ci_lower,
            "ci_upper": ci_upper,
        })

df_main = pd.DataFrame(main_rows)
df_main = df_main.sort_values(["h", "N", "model"]).reset_index(drop=True)
df_main.to_csv(OUT_DIR / "main_results_table.csv", index=False)
print("Saved main_results_table.csv")
pivot = df_main.pivot_table(index="model", columns=["h", "N"], values="delta_ll")
print(pivot.to_string())

Computing fresh h=1 baseline results (corrected label)...
  h=1 N=10: marginal=-2.3027, additive=-2.2991, backoff=-2.3012
  h=1 N=20: marginal=-2.9959, additive=-2.9986, backoff=-2.9958
  h=1 N=35: marginal=-3.5549, additive=-3.5551, backoff=-3.5548
  h=1 N=55: marginal=-4.0075, additive=-4.0000, backoff=-4.0072
  Training h=1 N=10 state_cond (corrected label)...


  Training h=1 N=10 state_free (corrected label)...


  Training h=1 N=20 state_cond (corrected label)...


  Training h=1 N=20 state_free (corrected label)...


  Training h=1 N=35 state_cond (corrected label)...


  Training h=1 N=35 state_free (corrected label)...


Saved main_results_table.csv
h                    1                                       2                                       5                                       10                              
N                    10        20        35        55        10        20        35        55        10        20        35        55        10        20        35        55
model                                                                                                                                                                        
additive       0.003673 -0.002699 -0.000203  0.007440  0.318591  0.233837  0.149737  0.162769  0.059119  0.044189  0.031439  0.014684  0.016392  0.012638  0.003705  0.006401
backoff        0.001576  0.000102  0.000118  0.000299  0.318517  0.228755  0.151859  0.160118  0.058471  0.044551  0.036307  0.011189  0.014813  0.011033  0.003710  0.007115
marginal       0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000  0.0000

## Section F: MIR / Irreducible Entropy Floor

**Motivation:** The raw NLL is hard to interpret without context.
`H_irr` = H(Y|X) under the **empirical** conditional distribution on TRAIN is
the minimum achievable NLL if you knew the true P(Y|X) perfectly.
`MIR = (H_irr − NLL_model) / H_irr` measures how close the model is to this ceiling.

- MIR ≈ 0: model is near-oracle (small residual gap)
- MIR < 0: model NLL > H_irr (usual for neural models with finite data; regularisation pushes above the empirical floor)
- MIR > 0: model generalises *beyond* the empirical transition matrix (possible via F_t features)

Why MIR matters when signal is small:
1. When I(X;Y) ≈ 0 nats, H_irr ≈ H(Y) (marginal entropy) and even a perfect oracle
   gains nothing from state conditioning — MIR distinguishes this from a model that
   simply fails to learn a real signal.
2. The relative gap `|NLL_model − H_irr| / H_irr` scales out marginal entropy,
   enabling fair comparison across horizons with different marginal difficulty.
3. A persistent negative MIR across all N and h is diagnostic: it means the
   empirical transition matrix is *over-fit* to training data and the neural
   regularisation is doing real work.
4. Comparing MIR across depths (Section I) shows whether additional capacity
   improves on the irreducible ceiling or merely memorises training noise.
5. Combined with the learning curves (Section H), a saturating MIR as train
   fraction grows indicates the dataset is the binding constraint, not model capacity.

In [18]:
from scripts.eval import compute_H_irr_from_counts, compute_MIR

# Key configs for MIR analysis — use all (h,N) for which we have a count matrix
# from Section A (sparsity_data already computed above)
mir_configs = [(h, N) for h in cfg.horizons for N in cfg.n_bins_list
               if (h, N) in sparsity_data]

# ── Load best neural NLL for each config from main_results_table ──
df_main_local = pd.read_csv(OUT_DIR / "main_results_table.csv")

mir_rows = []
for (h, N) in mir_configs:
    C = sparsity_data[(h, N)]["C"]  # shape (N_XT, N_actual)
    H_irr = compute_H_irr_from_counts(C)

    # Marginal entropy of Y on train (for reference)
    y_tr = configs[(h, N)]["y_all"][splits[h]["idx_train"]]
    N_actual = configs[(h, N)]["N_actual"]
    marginal = np.bincount(y_tr, minlength=N_actual).astype(np.float64)
    marginal = (marginal + 1e-12) / (marginal.sum() + N_actual * 1e-12)
    H_marginal = float(-(marginal * np.log(marginal)).sum())

    # For each model type that appears in main_results_table
    sub = df_main_local[(df_main_local["h"] == h) & (df_main_local["N"] == N)]
    for _, row in sub.iterrows():
        nll_model = -float(row["test_ll"])  # NLL = -mean_ll (positive)
        mir = compute_MIR(nll_model, H_irr)
        mir_rows.append({
            "h": h, "N": N, "model": row["model"],
            "test_ll": float(row["test_ll"]),
            "nll_model": nll_model,
            "H_irr": H_irr,
            "H_marginal": H_marginal,
            "MIR": mir,
        })

df_mir = pd.DataFrame(mir_rows).sort_values(["h", "N", "model"]).reset_index(drop=True)
df_mir.to_csv(OUT_DIR / "mir_table.csv", index=False)
print("Saved mir_table.csv")
print(df_mir[["h", "N", "model", "H_irr", "H_marginal", "nll_model", "MIR"]].to_string(index=False))

Saved mir_table.csv
 h  N         model    H_irr  H_marginal  nll_model       MIR
 1 10      additive 2.112358    2.302581   2.299062 -0.088387
 1 10       backoff 2.112358    2.302581   2.301159 -0.089379
 1 10      marginal 2.112358    2.302581   2.302735 -0.090126
 1 10 state_cond_nn 2.112358    2.302581   2.351315 -0.113123
 1 10 state_free_nn 2.112358    2.302581   2.304791 -0.091099
 1 20      additive 2.593342    2.995721   2.998606 -0.156271
 1 20       backoff 2.593342    2.995721   2.995806 -0.155191
 1 20      marginal 2.593342    2.995721   2.995907 -0.155230
 1 20 state_cond_nn 2.593342    2.995721   3.002134 -0.157631
 1 20 state_free_nn 2.593342    2.995721   2.992585 -0.153949
 1 35      additive 2.865977    3.555300   3.555118 -0.240456
 1 35       backoff 2.865977    3.555300   3.554797 -0.240344
 1 35      marginal 2.865977    3.555300   3.554916 -0.240385
 1 35 state_cond_nn 2.865977    3.555300   3.552295 -0.239471
 1 35 state_free_nn 2.865977    3.555300   3.55509

## Section G: Prefix Learning Curves (Consistency Check)

Train `state_cond` and `state_free` on expanding TRAIN prefixes (chronological),
evaluate on fixed val/test.  Key configs only: (h=1,N=55) and (h=10,N=55).

**What to look for:**
- **Decreasing NLL** as train_frac grows → model is data-limited (needs more data)
- **Saturation** early → model capacity or signal is the binding constraint, not data size
- **Divergence** (val NLL rises while train NLL falls) → overfitting sets in

A curve that saturates before train_frac=1.0 implies the dataset as a whole is sufficient
at the current model size; more data would not help unless the model is also scaled.

In [19]:
from scripts.plotting import plot_learning_curve
from torch.utils.data import DataLoader
from scripts.train import MasterDataset

LC_CONFIGS = [(1, 55), (10, 55)]
LC_FRACS   = [0.25, 0.50, 0.75, 1.00]
LC_SEEDS   = [42] if FAST_MODE else [42, 7, 123]
LC_MODELS  = ["state_cond", "state_free"]

lc_rows = []

for (h, N) in LC_CONFIGS:
    cfg_key = (h, N)
    if cfg_key not in configs:
        print(f"  Skipping (h={h}, N={N}) — not in configs")
        continue
    c = configs[cfg_key]
    N_actual = c["N_actual"]
    idx_train_full = splits[h]["idx_train"]
    idx_val        = splits[h]["idx_val"]
    idx_test       = splits[h]["idx_test"]

    # Fixed val/test loaders
    val_ds  = MasterDataset(F_normed, X_t_all, c["y_all"], idx_val)
    test_ds = MasterDataset(F_normed, X_t_all, c["y_all"], idx_test)
    val_loader_fixed  = DataLoader(val_ds,  batch_size=cfg.batch_eval, shuffle=False)
    test_loader_fixed = DataLoader(test_ds, batch_size=cfg.batch_eval, shuffle=False)

    for frac in LC_FRACS:
        n_tr = max(1, int(frac * len(idx_train_full)))
        idx_prefix = idx_train_full[:n_tr]
        tr_ds = MasterDataset(F_normed, X_t_all, c["y_all"], idx_prefix)
        tr_loader = DataLoader(tr_ds, batch_size=cfg.batch_train, shuffle=True)

        for model_type in LC_MODELS:
            for seed in LC_SEEDS:
                cache_name = f"lc_{model_type}_h{h}_N{N}_frac{int(frac*100)}_seed{seed}.pt"
                weight_path = CACHE_DIR / cache_name

                torch.manual_seed(seed)
                if model_type == "state_cond":
                    m = StateConditionedNet(n_features, N_XT, N_actual,
                                            hidden_dims=cfg.hidden_dims, dropout=cfg.dropout)
                else:
                    m = StateFreeNet(n_features, N_actual,
                                     hidden_dims=cfg.hidden_dims, dropout=cfg.dropout)

                if is_cached(weight_path):
                    load_cached_model(m, weight_path)
                else:
                    print(f"  Training LC h={h} N={N} {model_type} frac={frac:.2f} seed={seed}...")
                    best_state, _ = train_one_run(
                        m, tr_loader, val_loader_fixed, N_actual, c["sigma"], DEVICE,
                        lr=cfg.lr, weight_decay=cfg.weight_decay,
                        max_epochs=cfg.max_epochs, patience=cfg.patience,
                        grad_clip=cfg.grad_clip,
                    )
                    cache_model(best_state, weight_path)

                res_val  = evaluate_model(m, val_loader_fixed,  N_actual, DEVICE)
                res_test = evaluate_model(m, test_loader_fixed, N_actual, DEVICE)
                lc_rows.append({
                    "model": model_type, "h": h, "N": N,
                    "train_frac": frac, "seed": seed,
                    "nll_val":  -res_val["mean_ll"],
                    "nll_test": -res_test["mean_ll"],
                })

df_lc = pd.DataFrame(lc_rows)
df_lc.to_csv(OUT_DIR / "learning_curve_table.csv", index=False)
print("Saved learning_curve_table.csv")
print(df_lc.to_string(index=False))

  Training LC h=1 N=55 state_cond frac=0.25 seed=42...
  Training LC h=1 N=55 state_cond frac=0.25 seed=7...


  Training LC h=1 N=55 state_cond frac=0.25 seed=123...
  Training LC h=1 N=55 state_free frac=0.25 seed=42...


  Training LC h=1 N=55 state_free frac=0.25 seed=7...
  Training LC h=1 N=55 state_free frac=0.25 seed=123...


  Training LC h=1 N=55 state_cond frac=0.50 seed=42...
  Training LC h=1 N=55 state_cond frac=0.50 seed=7...


  Training LC h=1 N=55 state_cond frac=0.50 seed=123...
  Training LC h=1 N=55 state_free frac=0.50 seed=42...


...

  Training LC h=10 N=55 state_cond frac=1.00 seed=123...


  Training LC h=10 N=55 state_free frac=1.00 seed=42...


  Training LC h=10 N=55 state_free frac=1.00 seed=7...


  Training LC h=10 N=55 state_free frac=1.00 seed=123...


Saved learning_curve_table.csv
     model  h  N  train_frac  seed  nll_val  nll_test
state_cond  1 55        0.25    42 4.006375  4.014576
state_cond  1 55        0.25     7 4.004937  4.005221
state_cond  1 55        0.25   123 4.014574  4.010039
state_free  1 55        0.25    42 4.005935  4.012117
state_free  1 55        0.25     7 4.017984  4.011696
state_free  1 55        0.25   123 4.019086  4.009770
state_cond  1 55        0.50    42 4.005092  4.012860
state_cond  1 55        0.50     7 4.001961  4.003824
state_cond  1 55        0.50   123 4.013218  4.010037
state_free  1 55        0.50    42 4.005038  4.011429
state_free  1 55        0.50     7 4.015172  4.014107
state_free  1 55        0.50   123 4.002775  4.056627
state_cond  1 55        0.75    42 3.972997  4.037063
state_cond  1 55        0.75     7 3.964313  4.020062
state_cond  1 55        0.75   123 3.982793  4.000765
state_free  1 55        0.75    42 3.967396  4.166898
state_free  1 55        0.75     7 3.962189  4.1071

In [20]:
# Plot learning curves
for (h, N) in LC_CONFIGS:
    sub = df_lc[(df_lc["h"] == h) & (df_lc["N"] == N)]
    if len(sub) == 0:
        continue
    fig_lc = plot_learning_curve(sub, h, N, FIG_DIR)
    plt.close(fig_lc)
    print(f"  Saved learning_curve_nll_{h}_{N}.png/pdf")

  Saved learning_curve_nll_1_55.png/pdf


  Saved learning_curve_nll_10_55.png/pdf


## Section H: Depth Ablation (1-layer vs 5-layer, parameter-matched)

Compare shallow (1-hidden-layer) vs deep (5-hidden-layer, current default) architectures
for `state_cond` and `state_free` on (h=1,N=55) and (h=10,N=55).

The shallow hidden dimension H is chosen so that the shallow model's total parameter
count approximately matches the deep model, isolating depth as the variable.

In [21]:
from scripts.models import count_params, shallow_matched_hidden_dim
from scripts.plotting import plot_depth_ablation_bar

DA_CONFIGS = [(1, 55), (10, 55)]
DA_SEEDS   = [42] if FAST_MODE else [42, 7, 123]
DA_MODELS  = ["state_cond", "state_free"]
DEEP_DIMS  = cfg.hidden_dims  # (64,128,256,128,64)

# ── Compute parameter-matched shallow hidden dim once per (model_type, N) ──
shallow_h_map = {}  # (model_type, N) -> H
for (h_cfg, N) in DA_CONFIGS:
    cfg_key = (h_cfg, N)
    if cfg_key not in configs:
        continue
    N_actual = configs[cfg_key]["N_actual"]
    for model_type in DA_MODELS:
        key = (model_type, N_actual)
        if key in shallow_h_map:
            continue
        if model_type == "state_cond":
            deep_m = StateConditionedNet(n_features, N_XT, N_actual,
                                         hidden_dims=DEEP_DIMS, dropout=cfg.dropout)
            target = count_params(deep_m)
            H = shallow_matched_hidden_dim(
                StateConditionedNet, target,
                n_feat=n_features, n_xt_states=N_XT, n_output=N_actual, dropout=cfg.dropout
            )
        else:
            deep_m = StateFreeNet(n_features, N_actual,
                                   hidden_dims=DEEP_DIMS, dropout=cfg.dropout)
            target = count_params(deep_m)
            H = shallow_matched_hidden_dim(
                StateFreeNet, target,
                n_feat=n_features, n_output=N_actual, dropout=cfg.dropout
            )
        shallow_h_map[key] = H
        if model_type == "state_cond":
            shallow_p = count_params(StateConditionedNet(n_features, N_XT, N_actual,
                                                          hidden_dims=(H,), dropout=cfg.dropout))
        else:
            shallow_p = count_params(StateFreeNet(n_features, N_actual,
                                                   hidden_dims=(H,), dropout=cfg.dropout))
        print(f"  {model_type} N={N_actual}: deep params={target:,}  "
              f"shallow H={H} params={shallow_p:,}")

da_rows = []

for (h, N) in DA_CONFIGS:
    cfg_key = (h, N)
    if cfg_key not in configs:
        continue
    c = configs[cfg_key]
    N_actual = c["N_actual"]
    train_loader_full, val_loader_da, test_loader_da = build_loaders(
        c, F_normed, X_t_all,
        batch_train=cfg.batch_train, batch_eval=cfg.batch_eval,
    )

    archs = {
        "deep":    DEEP_DIMS,
        "shallow": None,  # filled per model_type
    }

    for model_type in DA_MODELS:
        shallow_H = shallow_h_map[(model_type, N_actual)]
        archs["shallow"] = (shallow_H,)

        for arch_name, hdims in archs.items():
            for seed in DA_SEEDS:
                cache_name = f"depth_{arch_name}_{model_type}_h{h}_N{N}_seed{seed}.pt"
                weight_path = CACHE_DIR / cache_name

                torch.manual_seed(seed)
                if model_type == "state_cond":
                    m = StateConditionedNet(n_features, N_XT, N_actual,
                                            hidden_dims=hdims, dropout=cfg.dropout)
                else:
                    m = StateFreeNet(n_features, N_actual,
                                     hidden_dims=hdims, dropout=cfg.dropout)

                if is_cached(weight_path):
                    load_cached_model(m, weight_path)
                else:
                    print(f"  Training DA h={h} N={N} {model_type} {arch_name} seed={seed}...")
                    best_state, _ = train_one_run(
                        m, train_loader_full, val_loader_da, N_actual, c["sigma"], DEVICE,
                        lr=cfg.lr, weight_decay=cfg.weight_decay,
                        max_epochs=cfg.max_epochs, patience=cfg.patience,
                        grad_clip=cfg.grad_clip,
                    )
                    cache_model(best_state, weight_path)

                res = evaluate_model(m, test_loader_da, N_actual, DEVICE)
                da_rows.append({
                    "model": model_type, "arch": arch_name, "h": h, "N": N,
                    "seed": seed, "nll_test": -res["mean_ll"],
                    "n_params": count_params(m),
                })

df_da = pd.DataFrame(da_rows)

# Compute delta_nll_vs_shallow = nll_deep - nll_shallow per (model,h,N,seed)
def _delta(grp):
    s = grp[grp["arch"] == "shallow"]["nll_test"].mean()
    grp = grp.copy()
    grp["delta_nll_vs_shallow"] = grp["nll_test"] - s
    return grp

df_da = df_da.groupby(["model", "h", "N", "seed"], group_keys=False).apply(_delta)
df_da = df_da.sort_values(["h", "N", "model", "arch", "seed"]).reset_index(drop=True)
df_da.to_csv(OUT_DIR / "depth_ablation_table.csv", index=False)
print("Saved depth_ablation_table.csv")
print(df_da[["h","N","model","arch","seed","nll_test","delta_nll_vs_shallow","n_params"]].to_string(index=False))

  state_cond N=55: deep params=102,071  shallow H=334 params=101,925
  state_free N=55: deep params=98,551  shallow H=394 params=98,555
  Training DA h=1 N=55 state_cond deep seed=42...


  Training DA h=1 N=55 state_cond deep seed=7...


  Training DA h=1 N=55 state_cond deep seed=123...


  Training DA h=1 N=55 state_cond shallow seed=42...


  Training DA h=1 N=55 state_cond shallow seed=7...


...

  Training DA h=10 N=55 state_free shallow seed=42...


  Training DA h=10 N=55 state_free shallow seed=7...


  Training DA h=10 N=55 state_free shallow seed=123...


Saved depth_ablation_table.csv
 h  N      model    arch  seed  nll_test  delta_nll_vs_shallow  n_params
 1 55 state_cond    deep     7  4.015749             -0.275504    102071
 1 55 state_cond    deep    42  4.014722             -0.437583    102071
 1 55 state_cond    deep   123  4.043311             -0.290456    102071
 1 55 state_cond shallow     7  4.291253              0.000000    101925
 1 55 state_cond shallow    42  4.452305              0.000000    101925
 1 55 state_cond shallow   123  4.333767              0.000000    101925
 1 55 state_free    deep     7  4.026646             -0.376423     98551
 1 55 state_free    deep    42  4.052250             -0.327861     98551
 1 55 state_free    deep   123  4.003799             -0.222576     98551
 1 55 state_free shallow     7  4.403069              0.000000     98555
 1 55 state_free shallow    42  4.380111              0.000000     98555
 1 55 state_free shallow   123  4.226375              0.000000     98555
10 55 state_cond    

/var/folders/xp/8yb64l_j1wd31jcqx1l7l_wr0000gp/T/ipykernel_15357/1790972654.py:109: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_da = df_da.groupby(["model", "h", "N", "seed"], group_keys=False).apply(_delta)


In [22]:
# Plot depth ablation bar charts
for (h, N) in DA_CONFIGS:
    sub = df_da[(df_da["h"] == h) & (df_da["N"] == N)]
    if len(sub) == 0:
        continue
    fig_da = plot_depth_ablation_bar(sub, h, N, FIG_DIR)
    plt.close(fig_da)
    print(f"  Saved depth_ablation_bar_{h}_{N}.png/pdf")

  Saved depth_ablation_bar_1_55.png/pdf
  Saved depth_ablation_bar_10_55.png/pdf


## Section I: Generalization Gap & Spectral Norm Proxy (Optional)

Train one key model (`state_cond`, h=1, N=55) with per-epoch NLL logging enabled.
Plot train vs val NLL curves and the generalization gap (val − train) per epoch.
Compute the spectral norm product of the best model as a capacity proxy.

In [23]:
from scripts.eval import compute_spectral_norm_product
from scripts.plotting import plot_gen_gap

GG_CONFIG = (1, 55)   # single key config for gen-gap experiment
h_gg, N_gg = GG_CONFIG

if GG_CONFIG in configs:
    c_gg = configs[GG_CONFIG]
    N_actual_gg = c_gg["N_actual"]
    tr_loader_gg, val_loader_gg, te_loader_gg = build_loaders(
        c_gg, F_normed, X_t_all,
        batch_train=cfg.batch_train, batch_eval=cfg.batch_eval,
    )

    gg_rows = []
    gg_figures = {}

    for model_type in ["state_cond", "state_free"]:
        cache_name = f"gengap_{model_type}_h{h_gg}_N{N_gg}_seed42.pt"
        weight_path = CACHE_DIR / cache_name
        hist_path   = CACHE_DIR / f"gengap_hist_{model_type}_h{h_gg}_N{N_gg}_seed42.npy"

        torch.manual_seed(42)
        if model_type == "state_cond":
            m_gg = StateConditionedNet(n_features, N_XT, N_actual_gg,
                                       hidden_dims=cfg.hidden_dims, dropout=cfg.dropout)
        else:
            m_gg = StateFreeNet(n_features, N_actual_gg,
                                 hidden_dims=cfg.hidden_dims, dropout=cfg.dropout)

        if is_cached(weight_path) and is_cached(hist_path):
            load_cached_model(m_gg, weight_path)
            nll_hist = dict(np.load(hist_path, allow_pickle=True).item())
            print(f"  Loaded gengap cache for {model_type}")
        else:
            print(f"  Training gen-gap model: {model_type} h={h_gg} N={N_gg}...")
            _, full_hist = train_one_run(
                m_gg, tr_loader_gg, val_loader_gg, N_actual_gg, c_gg["sigma"], DEVICE,
                lr=cfg.lr, weight_decay=cfg.weight_decay,
                max_epochs=cfg.max_epochs, patience=cfg.patience,
                grad_clip=cfg.grad_clip, log_nll=True,
            )
            cache_model({k: v.cpu() for k, v in m_gg.state_dict().items()}, weight_path)
            nll_hist = {k: v for k, v in full_hist.items() if k in ("train_nll", "val_nll")}
            np.save(hist_path, nll_hist)

        # Spectral norm product
        spec_prod = compute_spectral_norm_product(m_gg)
        print(f"  {model_type}: spectral_norm_product = {spec_prod:.4f}")

        if "train_nll" in nll_hist and len(nll_hist["train_nll"]) > 0:
            train_nll_arr = np.array(nll_hist["train_nll"])
            val_nll_arr   = np.array(nll_hist["val_nll"])
            gap_arr       = val_nll_arr - train_nll_arr
            figs_gg = plot_gen_gap(nll_hist, FIG_DIR, prefix=f"gengap_{model_type}_h{h_gg}_N{N_gg}")
            for f in figs_gg:
                plt.close(f)

            # Save per-epoch table
            for ep, (tr, va, gp) in enumerate(zip(train_nll_arr, val_nll_arr, gap_arr)):
                gg_rows.append({
                    "model": model_type, "epoch": ep + 1,
                    "train_nll": float(tr), "val_nll": float(va),
                    "gap": float(gp), "spec_prod": spec_prod,
                })

    if gg_rows:
        df_gg = pd.DataFrame(gg_rows)
        df_gg.to_csv(OUT_DIR / "generalization_gap_table.csv", index=False)
        print("Saved generalization_gap_table.csv")
        print(df_gg.groupby("model")[["gap", "spec_prod"]].describe().round(4))
else:
    print(f"Skipping Section J — config {GG_CONFIG} not available.")

  Training gen-gap model: state_cond h=1 N=55...


  state_cond: spectral_norm_product = 38.2833
  Training gen-gap model: state_free h=1 N=55...


  state_free: spectral_norm_product = 56.6550


Saved generalization_gap_table.csv
             gap                                                          \
           count    mean     std     min     25%     50%     75%     max   
model                                                                      
state_cond  20.0  0.1431  0.0652  0.0011  0.1062  0.1397  0.1989  0.2518   
state_free  31.0  0.0959  0.0325  0.0003  0.0752  0.1069  0.1207  0.1344   

           spec_prod                                                    \
               count     mean  std      min      25%      50%      75%   
model                                                                    
state_cond      20.0  38.2833  0.0  38.2833  38.2833  38.2833  38.2833   
state_free      31.0  56.6550  0.0  56.6550  56.6550  56.6550  56.6550   

                     
                max  
model                
state_cond  38.2833  
state_free  56.6550  


## Section J: Feature-Dimension Ablation (Overfitting Test)

**Goal:** Test whether reducing input feature dimensionality reduces overfitting
and improves generalisation.  With 194 features and only ~1,650 training samples,
the input space may be too wide for the model to generalise well even though the
training loss is low.

**Design:**
- **Configs:** (h=1, N=55) and (h=10, N=55); `StateConditionedNet` only.
- **Feature subsets:** Full (194), Top-50, Top-30, Top-15 — ranked by Mutual
  Information between each feature and Y_train, fitted on the training split
  only to prevent leakage.
- **Fixed:** architecture (`hidden_dims = cfg.hidden_dims`), seed, train/val/test
  split, label definition.

**What to look for:**
- If test NLL **falls** as features are removed → the high-dim input was hurting
  generalisation (overfitting hypothesis supported).
- If gen_gap **falls** with fewer features → smaller input dimension regularises
  implicitly.
- If spectral_product **falls** with fewer features → reduced capacity aligns with
  the shrinking gen_gap.
- If NLL is roughly flat → feature count is not the binding constraint.

In [24]:
from sklearn.feature_selection import mutual_info_classif
from scripts.eval import compute_spectral_norm_product
from scripts.plotting import plot_feature_ablation

FA_CONFIGS    = [(1, 55), (10, 55)]
FA_FEAT_SIZES = [15, 30, 50, 194]   # 194 = full feature set
FA_SEED       = cfg.seed             # single seed

# ── Pre-compute MI rankings once per (h, N) ──
# Keys: (h, N) -> np.ndarray of feature indices sorted descending by MI
mi_rankings = {}

for (h, N) in FA_CONFIGS:
    cfg_key = (h, N)
    if cfg_key not in configs:
        print(f"  Skipping MI ranking for (h={h}, N={N}) — not in configs")
        continue
    c = configs[cfg_key]
    idx_train_h = splits[h]["idx_train"]
    X_train_mi = F_normed[idx_train_h]            # (n_train, 194)
    y_train_mi = c["y_all"][idx_train_h]           # (n_train,)

    print(f"  Computing MI scores for h={h}, N={N} (n_train={len(idx_train_h)})...")
    mi_scores = mutual_info_classif(
        X_train_mi, y_train_mi, random_state=FA_SEED, n_neighbors=5
    )
    ranking = np.argsort(mi_scores)[::-1]          # descending
    mi_rankings[(h, N)] = ranking
    print(f"    Top-5 features by MI: {ranking[:5].tolist()}  "
          f"(scores: {mi_scores[ranking[:5]].round(4).tolist()})")

print("MI rankings computed.")

  Computing MI scores for h=1, N=55 (n_train=1656)...


    Top-5 features by MI: [156, 145, 153, 85, 190]  (scores: [0.0907, 0.0875, 0.0866, 0.0835, 0.0833])
  Computing MI scores for h=10, N=55 (n_train=1650)...


    Top-5 features by MI: [123, 23, 84, 94, 145]  (scores: [0.1698, 0.1689, 0.1689, 0.1686, 0.1683])
MI rankings computed.


In [25]:
fa_rows = []

for (h, N) in FA_CONFIGS:
    cfg_key = (h, N)
    if cfg_key not in configs or (h, N) not in mi_rankings:
        print(f"  Skipping (h={h}, N={N}) — configs or MI ranking missing")
        continue

    c        = configs[cfg_key]
    N_actual = c["N_actual"]
    ranking  = mi_rankings[(h, N)]

    # Fixed val/test loaders use FULL feature set only for index alignment;
    # we rebuild them per feature subset below.
    idx_train_h = splits[h]["idx_train"]
    idx_val_h   = splits[h]["idx_val"]
    idx_test_h  = splits[h]["idx_test"]

    for n_feat in sorted(FA_FEAT_SIZES):
        # Select top-n_feat features by MI (cap at actual number of features)
        n_feat_actual = min(n_feat, F_normed.shape[1])
        sel_idx = ranking[:n_feat_actual]
        F_sub   = F_normed[:, sel_idx]   # (T_full, n_feat_actual)

        cache_name  = f"feat_abl_n{n_feat_actual}_h{h}_N{N}_seed{FA_SEED}.pt"
        weight_path = CACHE_DIR / cache_name

        torch.manual_seed(FA_SEED)
        m_fa = StateConditionedNet(
            n_feat=n_feat_actual,
            n_xt_states=N_XT,
            n_output=N_actual,
            hidden_dims=cfg.hidden_dims,
            dropout=cfg.dropout,
        )

        # Build loaders using the feature-subset matrix F_sub
        tr_loader_fa, val_loader_fa, te_loader_fa = build_loaders(
            c, F_sub, X_t_all,
            batch_train=cfg.batch_train, batch_eval=cfg.batch_eval,
        )

        if is_cached(weight_path):
            load_cached_model(m_fa, weight_path)
            print(f"  Loaded  n_feat={n_feat_actual:3d} h={h} N={N} seed={FA_SEED}")
        else:
            print(f"  Training n_feat={n_feat_actual:3d} h={h} N={N} seed={FA_SEED}...")
            best_state, _ = train_one_run(
                m_fa, tr_loader_fa, val_loader_fa, N_actual, c["sigma"], DEVICE,
                lr=cfg.lr, weight_decay=cfg.weight_decay,
                max_epochs=cfg.max_epochs, patience=cfg.patience,
                grad_clip=cfg.grad_clip,
            )
            cache_model(best_state, weight_path)

        # ── Evaluate best checkpoint on all three splits ──
        # We need a train loader WITHOUT shuffle for consistent NLL
        from torch.utils.data import DataLoader
        train_eval_ds = MasterDataset(F_sub, X_t_all, c["y_all"], idx_train_h)
        train_eval_loader = DataLoader(
            train_eval_ds, batch_size=cfg.batch_eval, shuffle=False
        )

        res_train = evaluate_model(m_fa, train_eval_loader, N_actual, DEVICE)
        res_val   = evaluate_model(m_fa, val_loader_fa,    N_actual, DEVICE)
        res_test  = evaluate_model(m_fa, te_loader_fa,     N_actual, DEVICE)

        train_nll = -res_train["mean_ll"]
        val_nll   = -res_val["mean_ll"]
        test_nll  = -res_test["mean_ll"]
        gen_gap   = test_nll - train_nll
        spec_prod = compute_spectral_norm_product(m_fa)

        fa_rows.append({
            "horizon": h,
            "N_bins": N,
            "n_features": n_feat_actual,
            "train_nll": train_nll,
            "val_nll": val_nll,
            "test_nll": test_nll,
            "gen_gap": gen_gap,
            "spectral_product": spec_prod,
        })
        print(f"    train_nll={train_nll:.4f}  val_nll={val_nll:.4f}  "
              f"test_nll={test_nll:.4f}  gap={gen_gap:.4f}  spec={spec_prod:.3f}")

df_fa = (pd.DataFrame(fa_rows)
           .sort_values(["horizon", "N_bins", "n_features"])
           .reset_index(drop=True))
df_fa.to_csv(OUT_DIR / "feature_ablation_table.csv", index=False)
print("\nSaved feature_ablation_table.csv")
print(df_fa.to_string(index=False))

  Training n_feat= 15 h=1 N=55 seed=42...


    train_nll=3.9257  val_nll=3.9889  test_nll=4.0105  gap=0.0848  spec=65.065
  Training n_feat= 30 h=1 N=55 seed=42...


    train_nll=4.0073  val_nll=4.0047  test_nll=4.0036  gap=-0.0037  spec=2.170
  Training n_feat= 50 h=1 N=55 seed=42...


    train_nll=3.9146  val_nll=3.9966  test_nll=4.0289  gap=0.1143  spec=36.122
  Training n_feat=194 h=1 N=55 seed=42...


    train_nll=3.8904  val_nll=3.9773  test_nll=4.0191  gap=0.1287  spec=47.169
  Training n_feat= 15 h=10 N=55 seed=42...


    train_nll=3.9180  val_nll=3.9834  test_nll=3.9892  gap=0.0712  spec=87.178
  Training n_feat= 30 h=10 N=55 seed=42...


    train_nll=3.8907  val_nll=3.9832  test_nll=4.0045  gap=0.1138  spec=97.967
  Training n_feat= 50 h=10 N=55 seed=42...


    train_nll=3.9222  val_nll=3.9962  test_nll=4.0069  gap=0.0847  spec=30.019
  Training n_feat=194 h=10 N=55 seed=42...


    train_nll=3.9261  val_nll=3.9959  test_nll=3.9838  gap=0.0577  spec=10.617

Saved feature_ablation_table.csv
 horizon  N_bins  n_features  train_nll  val_nll  test_nll   gen_gap  spectral_product
       1      55          15   3.925683 3.988921  4.010481  0.084798         65.065065
       1      55          30   4.007314 4.004747  4.003594 -0.003720          2.169926
       1      55          50   3.914570 3.996561  4.028880  0.114310         36.122487
       1      55         194   3.890385 3.977322  4.019053  0.128669         47.168863
      10      55          15   3.918020 3.983416  3.989226  0.071207         87.177930
      10      55          30   3.890687 3.983165  4.004483  0.113796         97.967401
      10      55          50   3.922181 3.996228  4.006871  0.084691         30.018599
      10      55         194   3.926116 3.995911  3.983835  0.057718         10.617411


In [26]:
# Plot feature ablation: one figure per (h, N) config
for (h, N) in FA_CONFIGS:
    sub_fa = df_fa[(df_fa["horizon"] == h) & (df_fa["N_bins"] == N)]
    if len(sub_fa) == 0:
        print(f"  No data for h={h} N={N} — skipping plot")
        continue
    fig_fa = plot_feature_ablation(sub_fa, h, N, FIG_DIR)
    plt.close(fig_fa)
    print(f"  Saved feature_ablation_h{h}_N{N}.png/pdf")

  Saved feature_ablation_h1_N55.png/pdf
  Saved feature_ablation_h10_N55.png/pdf


## Summary

In [27]:
import json

def _load_table(path):
    try:
        return pd.read_csv(path)
    except Exception:
        return pd.DataFrame()

summary_lines = []

summary_lines.append("# MathFrameworkExperiments — Summary")
summary_lines.append(f"\n**Date:** {cfg.date_stamp}  \n**Git hash:** {cfg.git_hash}\n")

summary_lines.append("---\n")
summary_lines.append("## (i) Degeneracy Evidence\n")
summary_lines.append(
    "Transition degeneracy is diagnosed at the **cell level**: with ~1,650 training days "
    "and a 55×55 state-to-state space (3,025 possible transitions), "
    "count-based estimation is severely under-determined. The metrics below quantify this:\n\n"
    "- `frac_cells_zero` — fraction of joint-count cells C[i,j] that are exactly zero\n"
    "- `frac_cells_lt5` — fraction of cells with fewer than 5 observations\n"
    "- `median_nonzero_per_row` — median distinct output bins reached per input state\n"
    "- `p90_nonzero_per_row` — 90th percentile of same\n"
    "- `median_row_entropy_empirical` — median entropy of empirical row distributions\n"
    "- `median_row_maxprob_empirical` — median peak probability per row\n"
)
df_s = _load_table(OUT_DIR / "degeneracy_transition_table.csv")
if len(df_s):
    summary_lines.append(df_s.to_markdown(index=False))
    cum = df_s[df_s["config_type"] == "cumulative"] if "config_type" in df_s.columns else df_s
    if "frac_cells_lt5" in cum.columns and len(cum):
        v_lt5  = cum["frac_cells_lt5"].mean()
        v_zero = cum["frac_cells_zero"].mean()
        v_mnz  = cum["median_nonzero_per_row"].mean()
        summary_lines.append(
            f"\nFor cumulative configs, on average **{v_lt5*100:.1f}%** of cells C[i,j] "
            f"have fewer than 5 observations, and **{v_zero*100:.1f}%** are entirely unobserved. "
            f"The median number of nonzero cells per row is **{v_mnz:.1f}** out of "
            f"{int(cum['N_actual'].mean())} possible output bins.\n"
        )

summary_lines.append("---\n")
summary_lines.append("## (ii) Operator Diagnostics & Regime Case Study\n")
summary_lines.append(
    "Four diagnostics from the time-varying transition operator A_t^(1):\n\n"
    "- **Dobrushin coefficient** δ(A_t): contraction; spikes in high-volatility regimes.\n"
    "- **Row heterogeneity** ρ(A_t): state-dependence strength. Near-zero for StateFreeNet.\n"
    "- **Row entropy** H(A_t): higher = more uniform transitions.\n"
    "- **Spectral mixing proxy** σ_max(M): lower = faster mixing.\n"
)

summary_lines.append("---\n")
summary_lines.append("## (iii) Chapman–Kolmogorov Diagnostic Results\n")
summary_lines.append(
    "CK treated as a diagnostic. Label y_t^(h) := X_{t+h} in the same 55×55 space.\n\n"
    "**Ranking by CK consistency:** Backoff > StateFreeNet > StateConditionedNet. "
    "StateConditionedNet's deviation indicates the system is genuinely time-inhomogeneous "
    "and horizon-specific — not a model defect.\n"
)
df_ck_ = _load_table(OUT_DIR / "ck_table.csv")
if len(df_ck_):
    summary_lines.append(df_ck_.to_markdown(index=False))
    summary_lines.append("")

summary_lines.append("---\n")
summary_lines.append("## (iv) Uncertainty: Multi-Seed & Bootstrap CIs\n")
summary_lines.append(
    f"Ran {len(cfg.seeds)} seeds ({cfg.seeds}) for (h=1,N=55) and (h=10,N=55). "
    "Block bootstrap CIs (block_size=21, n_boot=500) on per-sample log-likelihood.\n\n"
    "**Label:** Y_t^(h) = bin((P_{t+1+h} - P_{t+1}) / P_{t+1}) — strictly forward-looking.\n"
)
key_configs = set(tuple(x) for x in cfg.bootstrap_key_configs)
df_m = _load_table(OUT_DIR / "main_results_table.csv")
if len(df_m):
    key_rows = df_m[df_m.apply(lambda r: (int(r["h"]), int(r["N"])) in key_configs, axis=1)].copy()
    if len(key_rows):
        summary_lines.append(key_rows[["h","N","model","test_ll","delta_ll","ci_lower","ci_upper"]].to_markdown(index=False))
        summary_lines.append("")

summary_lines.append("---\n")
summary_lines.append("## (v) MIR / Irreducible Entropy Floor\n")
summary_lines.append(
    "**H_irr** = H(Y|X) under empirical conditional on TRAIN. "
    "**MIR = (H_irr − NLL_model) / H_irr**.\n\n"
    "Why MIR matters when signal is small:\n"
    "1. When I(X;Y) ≈ 0, H_irr ≈ H(Y) — oracle gains nothing from state.\n"
    "2. MIR scales out marginal difficulty for cross-horizon comparison.\n"
    "3. Persistent negative MIR → neural regularisation works against over-fit empirical P.\n"
    "4. MIR across depths → capacity vs data bottleneck.\n"
    "5. MIR saturating as train_frac grows → data, not model, limits performance.\n"
)
df_mir_ = _load_table(OUT_DIR / "mir_table.csv")
if len(df_mir_):
    key_mir = df_mir_[df_mir_.apply(lambda r: (int(r["h"]), int(r["N"])) in key_configs, axis=1)]
    if len(key_mir):
        summary_lines.append(key_mir[["h","N","model","H_irr","H_marginal","nll_model","MIR"]].to_markdown(index=False))
        summary_lines.append("")
    mir_neg = (df_mir_["MIR"] < 0).mean()
    summary_lines.append(
        f"\nIn **{mir_neg*100:.0f}%** of configs, MIR < 0: neural regularisation "
        "improves test NLL beyond the empirically observed conditional entropy.\n"
    )

summary_lines.append("---\n")
summary_lines.append("## (vi) Prefix Learning Curves\n")
df_lc_ = _load_table(OUT_DIR / "learning_curve_table.csv")
if len(df_lc_):
    lc_summary = []
    for (h, N), grp in df_lc_.groupby(["h", "N"]):
        for model, mgrp in grp.groupby("model"):
            trend = mgrp.groupby("train_frac")["nll_test"].mean()
            delta = float(trend.iloc[-1] - trend.iloc[0])
            verdict = ("decreasing" if delta < -0.005
                       else ("saturating" if abs(delta) <= 0.005 else "increasing"))
            lc_summary.append({"h": h, "N": N, "model": model,
                                "nll_at_25pct": trend.iloc[0],
                                "nll_at_100pct": trend.iloc[-1],
                                "delta": delta, "trend": verdict})
    df_lcs = pd.DataFrame(lc_summary)
    summary_lines.append(df_lcs.to_markdown(index=False))
    summary_lines.append(
        "\n'decreasing' → still data-limited; 'saturating' → signal/capacity bound; "
        "'increasing' → overfitting warning.\n"
    )

summary_lines.append("---\n")
summary_lines.append("## (vii) Depth Ablation\n")
df_da_ = _load_table(OUT_DIR / "depth_ablation_table.csv")
if len(df_da_):
    agg_da = (df_da_.groupby(["h","N","model","arch"])[["nll_test","delta_nll_vs_shallow"]]
              .mean().reset_index())
    summary_lines.append(agg_da.to_markdown(index=False))
    deep_better = (df_da_["delta_nll_vs_shallow"] < 0).mean()
    summary_lines.append(
        f"\nDeep outperforms parameter-matched shallow in "
        f"**{deep_better*100:.0f}%** of (model, h, N, seed) combinations.\n"
    )

summary_lines.append("---\n")
summary_lines.append("## (viii) Generalization Gap & Spectral Norm\n")
df_gg_ = _load_table(OUT_DIR / "generalization_gap_table.csv")
if len(df_gg_):
    gg_agg = df_gg_.groupby("model")[["gap","spec_prod"]].agg(
        mean_gap=("gap","mean"), max_gap=("gap","max"), spec_prod=("spec_prod","first")
    ).reset_index()
    summary_lines.append(gg_agg.to_markdown(index=False))
    summary_lines.append(
        "\n**mean_gap** = average (val_nll − train_nll). Larger spec_prod → wider gap "
        "(consistent with PAC-Bayes theory).\n"
    )
else:
    summary_lines.append("*(Generalization gap table not found — Section J skipped.)*\n")

summary_lines.append("---\n")
summary_lines.append("## (ix) Feature-Dimension Ablation\n")
df_fa_ = _load_table(OUT_DIR / "feature_ablation_table.csv")
if len(df_fa_):
    summary_lines.append(df_fa_.to_markdown(index=False))
    summary_lines.append("")
    # Interpretation: check if reducing features helps
    for (h, N), grp in df_fa_.groupby(["horizon", "N_bins"]):
        grp = grp.sort_values("n_features")
        full_row = grp[grp["n_features"] == grp["n_features"].max()]
        best_row = grp.loc[grp["test_nll"].idxmin()]
        if len(full_row) == 0:
            continue
        full_nll  = float(full_row["test_nll"].values[0])
        best_nll  = float(best_row["test_nll"])
        best_nfeat = int(best_row["n_features"])
        delta_nll = best_nll - full_nll
        full_gap  = float(full_row["gen_gap"].values[0])
        best_gap  = float(best_row["gen_gap"])
        summary_lines.append(
            f"**h={h}, N={N}:** Best test NLL = {best_nll:.4f} at n_features={best_nfeat} "
            f"(full={full_nll:.4f}, Δ={delta_nll:+.4f}). "
            f"Gen-gap: full={full_gap:.4f} → best={best_gap:.4f}. "
        )
        if delta_nll < -0.005:
            summary_lines.append(
                "  ⟹ Reducing feature dimensionality **improves** test NLL: "
                "high-dim inputs were hurting generalisation.\n"
            )
        elif abs(delta_nll) <= 0.005:
            summary_lines.append(
                "  ⟹ Test NLL is **flat** across feature subsets: "
                "feature count is not the binding constraint here.\n"
            )
        else:
            summary_lines.append(
                "  ⟹ Full feature set **outperforms** subsets: "
                "all 194 features contribute useful signal.\n"
            )
else:
    summary_lines.append("*(Feature ablation table not found — Section K may have been skipped.)*\n")

summary_path = OUT_DIR / "summary.md"
summary_path.write_text("\n".join(summary_lines))
print(f"Saved summary.md ({len(summary_lines)} lines)")

Saved summary.md (46 lines)


In [28]:
import os
print(f"\n{'='*60}")
print(f"Results directory: {OUT_DIR}")
print(f"{'='*60}")
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        size = p.stat().st_size
        print(f"  {p.relative_to(OUT_DIR)}  ({size:,} bytes)")

expected = [
    "config.yaml",
    "degeneracy_label_table.csv",
    "degeneracy_transition_table.csv",
    "ck_table.csv",
    "calibration_table.csv",
    "main_results_table.csv",
    "mir_table.csv",
    "learning_curve_table.csv",
    "depth_ablation_table.csv",
    "feature_ablation_table.csv",
    "summary.md",
]
# h=10 figures only exist in full mode (FAST_MODE=False)
expected_figs = [
    "figures/learning_curve_nll_1_55.png",
    "figures/depth_ablation_bar_1_55.png",
    "figures/feature_ablation_h1_N55.png",
]
if not FAST_MODE:
    expected_figs += [
        "figures/learning_curve_nll_10_55.png",
        "figures/depth_ablation_bar_10_55.png",
        "figures/feature_ablation_h10_N55.png",
    ]

missing      = [f for f in expected      if not (OUT_DIR / f).exists()]
missing_figs = [f for f in expected_figs if not (OUT_DIR / f).exists()]

if missing:
    print(f"\nWARNING: Missing expected CSV/config files: {missing}")
else:
    print(f"\nAll {len(expected)} expected CSV/config files present.")

if missing_figs:
    print(f"WARNING: Missing expected figures: {missing_figs}")
else:
    print(f"All {len(expected_figs)} expected figures present.")

for opt in ["generalization_gap_table.csv"]:
    status = "present" if (OUT_DIR / opt).exists() else "absent"
    print(f"{opt}: {status}")

if not missing and not missing_figs:
    print("\nPipeline complete — all required outputs verified.")

# ── Final artifact verification (added by plan) ──
print("\n── Artifact verification ──")
_ck_path   = OUT_DIR / "ck_table.csv"
_degen_path = OUT_DIR / "degeneracy_transition_table.csv"
_heatmaps  = sorted(FIG_DIR.glob("At_heatmap_state_cond_t*.pdf"))
_checks_passed = True
if _ck_path.exists():
    _df_ck = pd.read_csv(_ck_path)
    _ok = len(_df_ck) == 12
    print(f"  ck_table.csv rows={len(_df_ck)} {'✓' if _ok else '✗ (expected 12)'}")
    _checks_passed &= _ok
else:
    print("  ck_table.csv MISSING ✗")
    _checks_passed = False
if _degen_path.exists():
    _df_degen = pd.read_csv(_degen_path)
    _ok_rows = len(_df_degen) >= 20
    _ok_cols = "frac_rows_lt5" in _df_degen.columns
    print(f"  degeneracy_transition_table.csv rows={len(_df_degen)} {'✓' if _ok_rows else '✗ (expected ≥20)'}, frac_rows_lt5 col: {'✓' if _ok_cols else '✗'}")
    _checks_passed &= _ok_rows and _ok_cols
else:
    print("  degeneracy_transition_table.csv MISSING ✗")
    _checks_passed = False
print(f"  At_heatmap_state_cond_t*.pdf count={len(_heatmaps)} {'✓' if len(_heatmaps) >= 3 else '✗ (expected ≥3)'}")
_checks_passed &= len(_heatmaps) >= 3
for _f in [
    "degeneracy_label_table.csv",
    "figures/sparsity_vs_N.pdf",
    "figures/transition_sparsity_heatmap.pdf",
    "figures/ck_error_summary.pdf",
]:
    _exists = (OUT_DIR / _f).exists()
    print(f"  {_f}: {'✓' if _exists else '✗ MISSING'}")
    _checks_passed &= _exists
if _checks_passed:
    print("\nAll artifact checks PASSED ✓")
else:
    print("\nSome artifact checks FAILED — see above ✗")



Results directory: /Users/JanRovirosaIlla/DeepMarkovResearch/results/paper_upgrade/2026-03-07
  cache/A_t_ck_state_cond_h1.npy  (28,640,828 bytes)
  cache/A_t_ck_state_cond_h10.npy  (28,531,928 bytes)
  cache/A_t_ck_state_cond_h2.npy  (28,628,728 bytes)
  cache/A_t_ck_state_cond_h5.npy  (28,592,428 bytes)
  cache/A_t_ck_state_free_h1.npy  (28,640,828 bytes)
  cache/A_t_ck_state_free_h10.npy  (28,531,928 bytes)
  cache/A_t_ck_state_free_h2.npy  (28,628,728 bytes)
  cache/A_t_ck_state_free_h5.npy  (28,592,428 bytes)
  cache/calib_state_cond_h10_N55_seed42.pt  (413,231 bytes)
  cache/calib_state_cond_h1_N55_seed42.pt  (413,213 bytes)
  cache/calib_state_free_h10_N55_seed42.pt  (399,151 bytes)
  cache/calib_state_free_h1_N55_seed42.pt  (399,133 bytes)
  cache/ck_state_cond_h10_seed42.pt  (413,105 bytes)
  cache/ck_state_cond_h1_seed42.pt  (413,087 bytes)
  cache/ck_state_cond_h2_seed42.pt  (413,087 bytes)
  cache/ck_state_cond_h5_seed42.pt  (413,087 bytes)
  cache/ck_state_free_h10_seed42